<a href="https://colab.research.google.com/github/EssaouriYassine/MapReduce_paradigm/blob/main/ESSAOURI_ZEDDAOUI_TP_MapReduce_exo_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Compte rendu du TD "Le Paradigme MapReduce" (2/4)

---

**Auteurs :** Yassine ESSAOURI, Oumaima ZEDDAOUI | **Encadrant :** Francois BASSAC

**Module :** Big Data | **Master 2 Cyber** | **Session :** 2026-2027 | **Rendu :** 28/09/2026 | **Statut :** Terminé

---

<br>

# Exercice 2 : Index Inversé (Inverted Index)

---

<br>

## 1. Objectif

L'objectif principal de cet exercice est de construire un **index inversé** à l'aide du paradigme **MapReduce**, structure de données fondamentale au cœur des moteurs de recherche.

Cet exercice permet de :
- Associer chaque mot présent dans un corpus de documents à la liste des identifiants de documents (`doc_id`) dans lesquels il apparaît.
- Gérer l'émission de clés représentant les mots et de valeurs représentant les identifiants de documents depuis la phase **Map**.
- Éliminer les doublons dans la phase **Reduce** pour garantir qu'un document n'est référencé qu'une seule fois par mot, même si le mot y apparaît plusieurs fois.

<br>

## 2. Problématique

- **Données d'entrée (Input) :** Un fichier CSV (`ReverseIndex.csv`) contenant des paires `(doc_id, doc_content)`.
- **Données de sortie (Output) :** Une liste ou dictionnaire de paires `(mot, [doc_id_1, doc_id_2, ...])` décrivant l'index inversé sans doublons de document.
- **Contraintes & Défis :**
  - Nettoyer et découper le contenu texte de chaque document (minuscules, ponctuation).
  - Traiter l'élimination des doublons au niveau du Reducer afin qu'un `doc_id` unique soit associé à un mot donné.

<br>

## 3. Mise en forme des sorties

Cette section conserve les mêmes fonctions de formatage et d'affichage standardisées pour garantir une présentation claire des résultats.

In [12]:
import os
import re
import csv

from typing import List, Tuple, Dict, Set

def print_separator(length: int = 50) -> None:
    # Affiche une ligne de séparation dans le terminal.
    print("-" * length)


def log_execution_start() -> None:
    # Affiche l'en-tête de lancement des commandes.
    print("▶ Lancement des commandes...\n")


def afficher_resultats(data, titre="RÉSULTAT", limite=10):
    # Normalisation en liste de paires (clé, valeur)
    if isinstance(data, dict):
        elements = list(data.items())
    else:
        elements = list(data)

    total = len(elements)
    reste = total - limite

    print(f"┌─ {titre.upper()} ─── ({total} éléments au total)")
    print("│")
    for i, (cle, valeur) in enumerate(elements[:limite], start=1):
        print(f"├─ {i:02d} — ({cle} , {valeur})")
    print("│")

    if reste > 0:
        print(f"└─ ... {reste} autres éléments masqués.")
    else:
        print(f"└─ Fin de la liste ({total} éléments).")

<br>

## 4. Démarche

La démarche suit l'enchaînement standard du modèle MapReduce adapté à la construction d'un index inversé :

### 4.1. **Préparation & Chargement des Données**
Chargement du fichier ReverseIndex.csv, lecture des colonnes doc_id et doc_content.   

In [13]:
print_separator()
log_execution_start()

# Chargement du fichier CSV
nom_fichier = 'ReverseIndex.csv'
documents = []

if os.path.exists(nom_fichier):
    print(f"Chargement direct du fichier : {nom_fichier}")
    chemin_final = nom_fichier
else:
    print(f"Fichier '{nom_fichier}' introuvable. Recherche d'une alternative...")
    fichiers = os.listdir('/content')
    fichier_trouve = next((f for f in fichiers if 'reverseindex' in f.lower()), None)

    if fichier_trouve is not None:
        print(f"Fichier trouvé : {fichier_trouve}")
        chemin_final = f'/content/{fichier_trouve}'
    else:
        raise FileNotFoundError("Aucun fichier ReverseIndex.csv n'a été trouvé.")

# Lecture du fichier CSV
with open(chemin_final, mode='r', encoding='utf-8') as f:
    lecteur = csv.reader(f)
    header = next(lecteur, None) # Saut de l'en-tête si présent
    for ligne in lecteur:
        if len(ligne) >= 2:
            doc_id, doc_content = ligne[0].strip(), ligne[1].strip()
            documents.append((doc_id, doc_content))

# Vérification du résultat
msg = f"Fichier chargé avec succès ! Nombre de documents : {len(documents)}"
print(msg)
print_separator(len(msg))

--------------------------------------------------
▶ Lancement des commandes...

Chargement direct du fichier : ReverseIndex.csv
Fichier chargé avec succès ! Nombre de documents : 6
----------------------------------------------------


#### Explication
Le test `if / else` : Vérifie l'existence du fichier ReverseIndex.csv dà la racine ou cherche un équivalent dans /content pour assurer une exécution fluide sur Colab. La lecture par csv.reader extrait les paires `(doc_id, doc_content)` nécessaires au traitement par le Mapper.

Fonctionnement : La fonction parcourt le texte du document, le passe en minuscules, extrait les mots sans la ponctuation, et émet la paire pour chaque occurrence.   

<br>

### 4.2. **Phase Map :**

* **Rôle :** Pour chaque document, extraire les mots nettoyés et émettre une paire `(mot, doc_id)`.

* **Entrée :** Une liste de tuples `(doc_id, doc_content)`.

* **Sortie :** Une liste de paires intermédiaires `(mot, doc_id)`.

* **Fonctionnement :** La fonction parcourt le texte du document, le passe en minuscules, extrait les mots sans la ponctuation, et émet la paire pour chaque occurrence.

In [14]:
def map_function(docs: List[Tuple[str, str]]) -> List[Tuple[str, str]]:
    paires: List[Tuple[str, str]] = []

    for doc_id, doc_content in docs:
        # Nettoyage et extraction des mots du document
        mots = re.findall(r'\b\w+\b', doc_content.lower())
        for mot in mots:
            paires.append((mot, doc_id))

    return paires

# Vérification du résultat
print_separator()
log_execution_start()

paires_map = map_function(documents)
afficher_resultats(paires_map, titre="Sortie Mapper", limite=10)
print_separator()

--------------------------------------------------
▶ Lancement des commandes...

┌─ SORTIE MAPPER ─── (61 éléments au total)
│
├─ 01 — (le , doc1.txt)
├─ 02 — (big , doc1.txt)
├─ 03 — (data , doc1.txt)
├─ 04 — (est , doc1.txt)
├─ 05 — (au , doc1.txt)
├─ 06 — (coeur , doc1.txt)
├─ 07 — (de , doc1.txt)
├─ 08 — (la , doc1.txt)
├─ 09 — (transformation , doc1.txt)
├─ 10 — (numerique , doc1.txt)
│
└─ ... 51 autres éléments masqués.
--------------------------------------------------


<br>

### 4.3. **Phase Shuffle & Sort (Simulée) :**

* **Rôle :** Regrouper tous les identifiants de documents associables à une même clé `(mot)`.  

* **Entrée :** La liste des paires intermédiaires `(mot, doc_id)`.

* **Sortie :** Un dictionnaire associant chaque mot à la liste de ses documents d'origine : `(mot, [doc_id_1, doc_id_2, doc_id_1...])`.

* **Fonctionnement :** Construit une liste de doc_id pour chaque mot clé rencontré.

In [15]:
def shuffle_and_sort(paires_map: List[Tuple[str, str]]) -> Dict[str, List[str]]:
    groupes = {}
    for mot, doc_id in paires_map:
        if mot not in groupes:
            groupes[mot] = []
        groupes[mot].append(doc_id)
    return groupes

# Vérification du résultat
print_separator()
log_execution_start()

groupes_shuffle = shuffle_and_sort(paires_map)
afficher_resultats(groupes_shuffle, titre="Shuffle & Sort", limite=10)
print_separator()

--------------------------------------------------
▶ Lancement des commandes...

┌─ SHUFFLE & SORT ─── (34 éléments au total)
│
├─ 01 — (le , ['doc1.txt', 'doc2.txt', 'doc2.txt', 'doc3.txt', 'doc4.txt', 'doc6.txt'])
├─ 02 — (big , ['doc1.txt', 'doc2.txt', 'doc3.txt', 'doc4.txt'])
├─ 03 — (data , ['doc1.txt', 'doc2.txt', 'doc3.txt', 'doc4.txt'])
├─ 04 — (est , ['doc1.txt', 'doc3.txt', 'doc4.txt', 'doc5.txt'])
├─ 05 — (au , ['doc1.txt'])
├─ 06 — (coeur , ['doc1.txt'])
├─ 07 — (de , ['doc1.txt', 'doc5.txt'])
├─ 08 — (la , ['doc1.txt', 'doc2.txt', 'doc6.txt'])
├─ 09 — (transformation , ['doc1.txt', 'doc2.txt', 'doc6.txt'])
├─ 10 — (numerique , ['doc1.txt'])
│
└─ ... 24 autres éléments masqués.
--------------------------------------------------


<br>

### 4.4. **Phase Reduce :**

* **Rôle :** Éliminer les doublons de `doc_id` associés à chaque mot pour former l'index inversé final.

* **Entrée :** Le dictionnaire regroupé `(mot, [doc_id_1, doc_id_2, doc_id_1...])`.

* **Sortie :** Un dictionnaire associant chaque mot à sa liste unique de documents : `(mot, [doc_id_1, doc_id_2...])`.

* **Fonctionnement :** Utilise la structure `set()` ou un parcours explicite pour ne conserver qu'une seule instance de chaque `doc_id` par mot.

In [16]:
def reduce_function(groupes_mots: Dict[str, List[str]]) -> Dict[str, List[str]]:
    resultat = {}
    for mot, liste_docs in groupes_mots.items():
        # Utilisation d'un ensemble pour éliminer les doublons, puis conversion en liste
        docs_uniques = []
        for doc_id in liste_docs:
            if doc_id not in docs_uniques:
                docs_uniques.append(doc_id)
        resultat[mot] = docs_uniques
    return resultat

# Vérification du résultat
print_separator()
log_execution_start()

resultat_final = reduce_function(groupes_shuffle)
afficher_resultats(resultat_final, titre="Résultat Reducer (Index Inversé)", limite=10)
print_separator()

--------------------------------------------------
▶ Lancement des commandes...

┌─ RÉSULTAT REDUCER (INDEX INVERSÉ) ─── (34 éléments au total)
│
├─ 01 — (le , ['doc1.txt', 'doc2.txt', 'doc3.txt', 'doc4.txt', 'doc6.txt'])
├─ 02 — (big , ['doc1.txt', 'doc2.txt', 'doc3.txt', 'doc4.txt'])
├─ 03 — (data , ['doc1.txt', 'doc2.txt', 'doc3.txt', 'doc4.txt'])
├─ 04 — (est , ['doc1.txt', 'doc3.txt', 'doc4.txt', 'doc5.txt'])
├─ 05 — (au , ['doc1.txt'])
├─ 06 — (coeur , ['doc1.txt'])
├─ 07 — (de , ['doc1.txt', 'doc5.txt'])
├─ 08 — (la , ['doc1.txt', 'doc2.txt', 'doc6.txt'])
├─ 09 — (transformation , ['doc1.txt', 'doc2.txt', 'doc6.txt'])
├─ 10 — (numerique , ['doc1.txt'])
│
└─ ... 24 autres éléments masqués.
--------------------------------------------------


<br>

### 5. **Conclusion**

L'implémentation de cet exercice montre l'application de MapReduce à la création d'un index inversé :

- **Inversion Clé/Valeur :** Le mapper transforme une relation `(Document -> Contenu)` en paires `(Mot -> Document)`, ce qui constitue l'étape fondamentale de l'indexation

- **Dédoublonnage dans Reduce :** Le Reducer garantit la concision de l'index en purgeant les récurrences multiples d'un même mot au sein d'un document donné.   

<br>

---

<br>

### Exportation du notebook

Cette section présente la procédure d'automatisation et de conversion du notebook au format HTML via Google Colab et `nbconvert`.

In [17]:
import os
from google.colab import files, drive

# 1. Montage du Drive
drive.mount('/content/drive', force_remount=True)

# 2. Nom du fichier à chercher
target_name = 'ESSAOURI-ZEDDAOUI-TP-MapReduce_exo_2.ipynb'
found_path = None

# Recherche récursive dans tout le Drive
print('Recherche du notebook dans votre Google Drive...')
for root, dirs, filenames in os.walk('/content/drive/MyDrive'):
    for filename in filenames:
        if filename.lower() == target_name.lower():
            found_path = os.path.join(root, filename)
            break
    if found_path:
        break

if found_path:
    print(f'✅ Fichier trouvé : {found_path}')

    # Conversions via nbconvert
    !jupyter nbconvert --to html "{found_path}"
    !jupyter nbconvert --to markdown "{found_path}"

    base_name = os.path.splitext(os.path.basename(found_path))[0]
    drive_dir = os.path.dirname(found_path)

    # Telechargement des fichiers générés (HTML & MD)
    for ext in ['.html', '.md']:
        filename = base_name + ext
        drive_path = os.path.join(drive_dir, filename)

        if os.path.exists(filename):
            files.download(filename)
        elif os.path.exists(drive_path):
            files.download(drive_path)
        else:
            print(f'⚠️ Impossible de localiser le fichier généré : {filename}')
else:
    print(
        '❌ Fichier introuvable sur Google Drive. Vérifiez le nom exact du'
        ' fichier.'
    )

Mounted at /content/drive
Recherche du notebook dans votre Google Drive...
✅ Fichier trouvé : /content/drive/MyDrive/Colab Notebooks/ESSAOURI-ZEDDAOUI-TP-MapReduce_exo_2.ipynb
[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/ESSAOURI-ZEDDAOUI-TP-MapReduce_exo_2.ipynb to html
[NbConvertApp] Writing 318558 bytes to /content/drive/MyDrive/Colab Notebooks/ESSAOURI-ZEDDAOUI-TP-MapReduce_exo_2.html
[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/ESSAOURI-ZEDDAOUI-TP-MapReduce_exo_2.ipynb to markdown
[NbConvertApp] Writing 12425 bytes to /content/drive/MyDrive/Colab Notebooks/ESSAOURI-ZEDDAOUI-TP-MapReduce_exo_2.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>